# Knowledge Distillation Demo on CIFAR-10

In this demo, we train a larger **teacher model** and a smaller **student model** on the CIFAR-10 dataset. We first train the student using standard cross-entropy loss, and then train another student using **knowledge distillation**, where the student learns from both the true labels and the soft predictions of the teacher.

The goal is to compare:
- Teacher model performance
- Student model performance without distillation
- Student model performance with distillation

## Import Libraries and Configure Device

We begin by importing the required PyTorch and torchvision libraries for building neural networks, loading datasets, and training models.

We also detect the available hardware accelerator (such as GPU, MPS, or CPU) so that training can run on the most suitable device.

In [1]:
from tqdm.auto import tqdm

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


## Load and Preprocess the CIFAR-10 Dataset

In this section, we define the preprocessing pipeline for CIFAR-10 images.

About CIFAR-10:

 - 60,000 images total
 - 10 classes (airplane, car, bird, etc.)
 - Each image = 32×32 RGB

The images are:
- converted to tensors
- normalized using predefined mean and standard deviation values

We then download and load the CIFAR-10 training and test datasets.

In [3]:
# Below we are preprocessing data for CIFAR-10.
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
# Loading the CIFAR-10 dataset:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

100%|██████████| 170M/170M [00:05<00:00, 30.2MB/s]


## Create DataLoaders

We use PyTorch DataLoaders to efficiently load the training and test data in batches.

Here:
- batch size is set to 128
- training data is shuffled
- test data is not shuffled

In [5]:
BATCH_SIZE = 1024

In [6]:
# Dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Define the Teacher and Student Networks

We define two convolutional neural networks:

- **DeepNN**: a larger and deeper model that will act as the **teacher**
- **LightNN**: a smaller model that will act as the **student**

The teacher has more layers and parameters, allowing it to learn richer representations.  
The student is designed to be lighter and more efficient, but may have lower accuracy unless guided by the teacher.

In [7]:
# Deeper neural network class to be used as teacher:
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [8]:
# Lightweight neural network class to be used as student:
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super(LightNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

## Define Training and Evaluation Functions

We now create helper functions for:
- training a model using cross-entropy loss
- evaluating a model on the test set

The `train()` function performs the standard training loop, while the `test()` function computes classification accuracy on the test data.

In [9]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in tqdm(range(epochs), desc="Epoch", leave=False):
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc="Batch", leave=False):
            # inputs: A collection of batch_size images
            # labels: A vector of dimensionality batch_size with integers denoting class of each image
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            # outputs: Output of the network for the collection of images. A tensor of dimensionality batch_size x num_classes
            # labels: The actual labels of the images. Vector of dimensionality batch_size
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [10]:
def test(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Batch", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

## Train the Teacher Model

We first train the larger teacher network on CIFAR-10 using standard supervised learning.

This trained teacher will later provide soft targets for knowledge distillation.

In [11]:
torch.manual_seed(42)

In [12]:
nn_deep = DeepNN(num_classes=10).to(device)

In [13]:
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 1/10, Loss: 1.7868588311331612


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 2/10, Loss: 1.3567945567929014


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

... (13 outputs left out)

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 10/10, Loss: 0.42392919683942987


Batch:   0%|          | 0/10 [00:00<?, ?it/s]

Test Accuracy: 73.06%


In [14]:
# Instantiate the lightweight network:
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)

## Initialize Student Models

Next, we initialize lightweight student models.

We create:
- `nn_light` for training with standard cross-entropy only
- `new_nn_light` for training with knowledge distillation

Both models start from the same random seed so that the comparison is fair.

In [15]:
torch.manual_seed(42)
new_nn_light = LightNN(num_classes=10).to(device)

In [16]:
# Print the norm of the first layer of the initial lightweight model
print("Norm of 1st layer of nn_light:", torch.norm(nn_light.features[0].weight).item())
# Print the norm of the first layer of the new lightweight model
print("Norm of 1st layer of new_nn_light:", torch.norm(new_nn_light.features[0].weight).item())

Norm of 1st layer of nn_light: 2.327361822128296
Norm of 1st layer of new_nn_light: 2.327361822128296


## Compare Model Parameter Counts

Before training the student, we compare the number of parameters in the teacher and student networks.

This gives a sense of the size difference between the two models and helps explain why the student is more efficient but potentially less accurate.

In [17]:
total_params_deep = "{:,}".format(sum(p.numel() for p in nn_deep.parameters()))
print(f"DeepNN parameters: {total_params_deep}")
total_params_light = "{:,}".format(sum(p.numel() for p in nn_light.parameters()))
print(f"LightNN parameters: {total_params_light}")

DeepNN parameters: 1,186,986
LightNN parameters: 267,738


## Train the Student Model with Cross-Entropy Loss

We now train the lightweight student model using only the true class labels and standard cross-entropy loss.

This serves as the baseline student model, which we will later compare against the distilled student.

In [18]:
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light_ce = test(nn_light, test_loader, device)

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 1/10, Loss: 1.8102401811249402


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 2/10, Loss: 1.4297786440168108


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

... (13 outputs left out)

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 10/10, Loss: 0.950025484269979


Batch:   0%|          | 0/10 [00:00<?, ?it/s]

Test Accuracy: 64.46%


In [19]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light_ce:.2f}%")

Teacher accuracy: 73.06%
Student accuracy: 64.46%


## Define the Knowledge Distillation Training Function

In knowledge distillation, the student learns from two sources:

1. **Hard targets**: the true labels using cross-entropy loss  
2. **Soft targets**: the teacher’s softened output distribution using KL divergence

A temperature parameter `T` is used to soften the logits, and the final loss is a weighted combination of:
- cross-entropy loss
- distillation loss

##Idea
Normally, a student model learns only from the correct labels.

For example: image = cat label = 3

So the student only learns: “Try to predict class 3.”

But in knowledge distillation, the student also learns from the teacher’s output distribution.

For example, for the same image, the teacher might say:

 - cat: 0.72
 - dog: 0.18
 - deer: 0.06
 - fox: 0.04

That is useful because it tells the student:

 - the image is mainly a cat
 - but it also looks a bit like a dog

This extra information is called soft knowledge.

##What is temperature?

Temperature controls how soft or smooth the output distribution becomes.

 - If T = 1 This is normal softmax.

 - If T > 1 The probabilities become softer and less extreme.

For example:
Without temperature:

- class A = 0.95
- class B = 0.03
- class C = 0.02

With higher temperature:

- class A = 0.60
- class B = 0.25
- class C = 0.15

Now the student can see that: class A is still best but B and C are also somewhat related

This extra information is useful.

In [20]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in tqdm(range(epochs), desc="Epochs", leave=False):
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc="Batch", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # Forward pass with the student model
            student_logits = student(inputs)

            #Soften the student logits by applying softmax first and log() second
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)

            # Calculate the soft targets loss. Scaled by T**2 as suggested by the authors of the paper "Distilling the knowledge in a neural network"
            # soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)
            soft_targets_loss = nn.functional.kl_div(
                soft_prob,
                soft_targets,
                reduction='batchmean'
            ) * (T ** 2)

            # Calculate the true label loss
            label_loss = ce_loss(student_logits, labels)

            # Weighted sum of the two losses
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [21]:
# Apply `train_knowledge_distillation` with a temperature of 2.
# Arbitrarily set the weights to 0.75 for CE and 0.25 for distillation loss.
train_knowledge_distillation(teacher=nn_deep, student=new_nn_light, train_loader=train_loader, epochs=10,
                             learning_rate=0.001, T=2, soft_target_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_light_ce_and_kd = test(new_nn_light, test_loader, device)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 1/10, Loss: 2.3328595818305504


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 2/10, Loss: 1.7675480623634494


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

... (13 outputs left out)

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 10/10, Loss: 1.0644330881079849


Batch:   0%|          | 0/10 [00:00<?, ?it/s]

Test Accuracy: 64.10%


In [22]:
# Compare the student test accuracy with and without the teacher, after distillation
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

Teacher accuracy: 73.06%
Student accuracy without teacher: 64.46%
Student accuracy with CE + KD: 64.10%


## Train the Student with Knowledge Distillation

We now train a new student model using both:
- cross-entropy loss with the true labels
- KL divergence loss with the teacher's softened predictions

In the first experiment:
- temperature = 2
- distillation loss weight = 0.25
- cross-entropy weight = 0.75

In this second experiement:
- temperature = 2
- distillation loss weight = 0.5
- cross-entropy weight = 0.5

We then compare the resulting accuracy with the baseline student.

In [23]:
# Reset student model from scratch
new_nn_light_1 = LightNN(num_classes=10).to(device)

In [24]:
# Apply `train_knowledge_distillation` with a temperature of 2.
# Arbitrarily set the weights to 0.5 for CE and 0.5 for distillation loss.
train_knowledge_distillation(teacher=nn_deep, student=new_nn_light_1, train_loader=train_loader, epochs=10,
                             learning_rate=0.001, T=2, soft_target_loss_weight=0.5, ce_loss_weight=0.5, device=device)
test_accuracy_light_ce_and_kd_1 = test(new_nn_light_1, test_loader, device)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 1/10, Loss: 2.7810317399550457


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 2/10, Loss: 2.002527560506548


Batch:   0%|          | 0/49 [00:00<?, ?it/s]

... (13 outputs left out)

Batch:   0%|          | 0/49 [00:00<?, ?it/s]

Epoch 10/10, Loss: 1.124741512901929


Batch:   0%|          | 0/10 [00:00<?, ?it/s]

Test Accuracy: 62.71%


In [25]:
# Compare the student test accuracy with and without the teacher, after distillation
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd_1:.2f}%")

Teacher accuracy: 73.06%
Student accuracy without teacher: 64.46%
Student accuracy with CE + KD: 62.71%
